In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from modules.coord_transform import *
from modules.simulation import *
from modules.likelihood import *
from modules.mcmc import run_mcmc  
from modules.localise import ci_68_and_sigma
from modules.constellation import get_satellite_positions,generate_full_constellation,get_pointing_radec,eci_to_latlon


import os
from matplotlib.patches import Patch

In [ ]:
def catalogue(type):
    if type =="short":
        cat = pd.read_csv("D:\\Unimelb\\SpIRIT_In_Polar_Orbit\\mcmc\\others\\short_grb_catalogue_filter.txt",sep ='|')
    elif type=="long":
        cat = pd.read_csv("D:\\Unimelb\\SpIRIT_In_Polar_Orbit\\mcmc\\others\\long_grb_catalogue_filter.txt", sep='|' )
    return cat

In [ ]:
np.random.seed(4)

Area = 50
offset_angle = [10]
planes = [
        {"incl": 0, "rot": 0},       # Equatorial
        {"incl": 100, "rot": 0},     # Polar
        {"incl": 50, "rot": 0},      # Diagonal 1 
        {"incl": 150, "rot": 0},     # Diagonal 2 
    ]
type ="long"
flux_limit = flux_lower_bound(type) # Change flux_limit for long grb = 0.463 and for short grb= 1.861
cat = catalogue(type)
df_results_ = {}
for i, offset in enumerate(offset_angle):
    localisation_result = []
    n_det = 0
    total_generate = 0  # Counter for total GRBs generated

    minimum_num_sats_detecting_burst = 3

    while n_det < 3: # change here to set the number of detection

        t_random = np.random.uniform(0, 96 * 60)

        sat_pos = generate_full_constellation(t_random, planes)
        lat_lon = eci_to_latlon(sat_pos, t_random)
        num_sats = len(sat_pos)

        pointings, base_dirs = get_pointing_radec(sat_pos, num_sats , offset_deg = offset_angle)
        sat_pointing = np.array([coord_transform.r2c(ra, dec) for ra, dec in pointings])

        grb_vec, ra, dec, t_90, flux_avg = generate_grb(cat)

        print(ra,dec)
        total_generate += 1

        grb_info = {"ra": ra, 
                "dec":dec, 
                "flux_avg":flux_avg, 
                "t_90": t_90}  
        det_result = simulate_satellite_det(grb_vec, flux_avg, sat_pos, sat_pointing, flux_limit, lat_lon)

        sat_info = {'Area': Area, 
                    'sat_pos' : sat_pos,
                'sat_pointing': sat_pointing, 
                'flux_limit': flux_limit,
                'offset': offset,
                'lat_lon':lat_lon}

        if sum( det_result[0] > 0 ) >=  minimum_num_sats_detecting_burst :  # Ensure it skips GRBs that do not meet conditions
        
            print(ra, dec)
            n_det += 1
            f_obs, t_obs= det_result
            print(f"Detection result:{det_result}")

            Ph_obs = np.array([ 0 if f < flux_limit 
                            else np.random.poisson(f * t_90 * Area) for f in f_obs])  # Observed Photon count
            
            save_path = os.path.join( f"./orbit_results/{num_sats}n_0_{type}_grb_plots", f"offset_{offset}", f"detection_{n_det}")
            
            obs_info = {'t_obs': t_obs,
                    'f_obs': f_obs,
                    'Ph_obs': Ph_obs }
        
            mcmc_params= {'steps': 7000, 
                      'nwalk': 8, 
                      'discard': 1000,
                      'move': 2, 
                      'save_path': save_path,
                      'corner_show' :False ,'corner_save' : True, 
                      'chain_show': False, 'chain_save' : True}
            
            #localise_params ={'localisation_show' :False, 'localisation_save' :True}
        
            flat_samples = run_mcmc(**grb_info,
                                        **obs_info,
                                        **sat_info,
                                        **mcmc_params)
            
            true_value = np.array([ra, dec, flux_avg])
            ci_area_68, containment, resolution = ci_68_and_sigma(flat_samples, true_value,
                                                   type,
                                                   save_path,
                                                   localisation_show=True,
                                                   localisation_save=True)
                                                        

            print(f'RA:{ra},DEC:{dec},FLUX:{flux_avg}')
            print(f"68% confidence region area: {ci_area_68:.2f} deg²")
            print(f"area of grid:{resolution}deg²")
            print(f'Localization completed for detection #{n_det}')    
        
            localisation_result.append([ra, dec, t_90, flux_avg, t_random, f_obs, np.count_nonzero(f_obs), ci_area_68, containment, total_generate])

        df_results_[i] = pd.DataFrame(localisation_result, columns=['RA', 'Dec', 'T90', 'Flux_Avg', 'Time', 'fcos','n_det' ,'CI_Area_68', 'containment','Total generated'])
    
df_results_[i].to_csv(f'./orbit_results/{num_sats}n_0_{type}_grb_plots/{type}_{offset}.csv', index=False)
#print(f'Total GRBs generated: {total_generate}, Total detected: {n_det}')